# 02 - Merge flats and houses

Concatenate the two cleaned tables. There is no source notebook for cleaning `houses.csv`; that lives only in `src/preprocessing/cleaning.clean_houses()`.

Notebook 03 edits sectors by hardcoded row position (`df.loc[955,'sector']=...`) against the committed `gurgaon_properties.csv`, so that file stays canonical - this notebook seeds its own shuffle and writes to `data/interim/_rebuild/` instead of feeding 03.

In [1]:
import numpy as np
import pandas as pd

In [2]:
flats = pd.read_csv('../data/interim/flats_cleaned.csv')
houses = pd.read_csv('../data/interim/house_cleaned.csv')

In [3]:
df = pd.concat([flats,houses],ignore_index=True)

In [4]:
df = df.sample(df.shape[0],ignore_index=True,random_state=42)  # seeded so this notebook's own output is reproducible

In [5]:
df.head()

,property_name,property_type,society,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,3 BHK Flat in Sector 65 Gurgaon,flat,m3m heights,2.50,13600.0,1838.0,Super Built up area 1828(169.83 sq.m.),3,3,3,not available,"Sector 65 Gurgaon, Gurgaon, Haryana",19.0,NaN,Dec 2024,"['Rapid Metro Sector 56', 'M3m 65th Avenue Mal...",We are the proud owners of this 3 bhk apartmen...,[],NaN,"['Environment4 out of 5', 'Safety4 out of 5', ..."
1,4 BHK Flat in Sector 48 Gurgaon,flat,bestech park view city,2.65,10323.0,2567.0,Super Built up area 2567(238.48 sq.m.)Carpet a...,4,4,3+,"study room,servant room","1101, Sector 48 Gurgaon, Gurgaon, Haryana",11.0,North-East,5 to 10 Year Old,"['Rapid Metro Sector 56', 'Sapphire Mall', 'Om...",This lovely 4 bhk apartment/flat in sector 48 ...,"['4 Wardrobe', '1 Water Purifier', '8 Fan', '1...","['Centrally Air Conditioned', 'Water purifier'...","['Green Area5 out of 5', 'Construction4 out of..."
2,2 BHK Flat in Sector 85 Gurgaon,flat,ss the leaf,1.20,7317.0,1640.0,Super Built up area 1640(152.36 sq.m.)Built Up...,2,2,3,not available,"704, Sector 85 Gurgaon, Gurgaon, Haryana",6.0,North,1 to 5 Year Old,"['Sapphire 83 Mall', 'Dwarka Expressway', 'Cen...",This lovely 2 bhk apartment/flat in sector 85 ...,"['3 Wardrobe', '9 Fan', '1 Exhaust Fan', '3 Ge...","['Water purifier', 'Centrally Air Conditioned'...","['Green Area4.5 out of 5', 'Construction4.5 ou..."
3,3 BHK Flat in Sector 107 Gurgaon,flat,signature global solera,0.52,8062.0,645.0,Carpet area: 645 (59.92 sq.m.),3,2,2,others,"Sector 107 Gurgaon, Gurgaon, Haryana",5.0,East,1 to 5 Year Old,"['Gurgaon Dreamz Mall', 'Dwarka Expressway', ""...","Situated in sector 107 gurgaon, signature glob...","['3 Wardrobe', '4 Fan', '6 Light', 'No AC', 'N...","['Intercom Facility', 'Lift(s)', 'Park']","['Green Area4.5 out of 5', 'Construction4.5 ou..."
4,2 BHK Flat in Sohna,flat,signature global park,0.54,7248.0,745.0,Carpet area: 745 (69.21 sq.m.),2,1,3,not available,"J-26, Sohna, Gurgaon, Haryana",1.0,NaN,Within 6 months,"['Sector 55-56 metro', 'Global city centre', '...",Park facing property for the unique view,"['1 Modular Kitchen', 'No AC', 'No Bed', 'No C...","['Feng Shui / Vaastu Compliant', 'Security / F...","['Green Area5 out of 5', 'Construction5 out of..."


In [6]:
import os
os.makedirs('../data/interim/_rebuild', exist_ok=True)
df.to_csv('../data/interim/_rebuild/gurgaon_properties.csv',index=False)  # sidecar; notebook 03 reads the canonical copy

---
### Check (added, not in the original): this run vs. the canonical file
Order-independent row comparison against the committed `data/interim/gurgaon_properties.csv`. Free-text columns are stripped of carriage returns first - the canonical file stores embedded newlines as `\r\n`, pandas writes `\n`; that encoding difference is not a data difference.

In [7]:
canonical = pd.read_csv('../data/interim/gurgaon_properties.csv')

def _norm(frame):
    out = frame.copy()
    for c in out.select_dtypes('object').columns:
        out[c] = out[c].str.replace('\r\n', '\n', regex=False).str.replace('\r', '\n', regex=False)
    return out

key = list(df.columns)
a = _norm(df).sort_values(key, kind='stable', na_position='first').reset_index(drop=True)
b = _norm(canonical).sort_values(key, kind='stable', na_position='first').reset_index(drop=True)
cell_eq = (a == b) | (a.isna() & b.isna())
print('this run shape / canonical shape          :', df.shape, '/', canonical.shape)
print('same columns                             :', key == list(canonical.columns))
print('same dtypes                              :', (df.dtypes.values == canonical.dtypes.values).all())
print('mismatching cells (order-independent)    :', int((~cell_eq).values.sum()))
print('row-for-row content-identical            :', bool(cell_eq.values.all()))

this run shape / canonical shape          : (3961, 20) / (3961, 20)
same columns                             : True
same dtypes                              : True
mismatching cells (order-independent)    : 0
row-for-row content-identical            : True
